In [1]:
from relbench.datasets import get_dataset
from relbench.tasks import get_task

dataset = get_dataset("rel-stack", download=True)
db = dataset.get_db()


/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Database object from /home/abed/.cache/relbench/rel-stack/db...
Done in 9.11 seconds.


In [2]:
task = get_task("rel-stack", "user-engagement", download=True)
train_table = task.get_table("train")
val_table   = task.get_table("val")
test_table  = task.get_table("test")

In [3]:
print("Tables in DB:", list(db.table_dict.keys()))
for name, tbl in db.table_dict.items():
    print(f"\n=== {name} ===  shape={tbl.df.shape}")
    print(tbl.df.dtypes)
    print(tbl.df.head(2))


Tables in DB: ['users', 'postHistory', 'postLinks', 'votes', 'posts', 'badges', 'comments']

=== users ===  shape=(255360, 8)
Id                          Int64
AccountId                 float64
DisplayName                object
Location                   object
ProfileImageUrl           float64
WebsiteUrl                 object
AboutMe                    object
CreationDate       datetime64[ns]
dtype: object
   Id  AccountId   DisplayName            Location  ProfileImageUrl  \
0   0       -1.0     Community  on the server farm              NaN   
1   1        2.0  Geoff Dalgas       Corvallis, OR              NaN   

                       WebsiteUrl  \
0  http://meta.stackexchange.com/   
1        http://stackoverflow.com   

                                             AboutMe            CreationDate  
0  <p>Hi, I'm not really a person.</p>\n\n<p>I'm ... 2010-07-19 06:55:26.860  
1  <p>Dev #2 who helped create Stack Overflow cur... 2010-07-19 14:01:36.697  

=== postHistory ===  sha

In [4]:
for name, t in [("train", train_table), ("val", val_table), ("test", test_table)]:
    df = t.df
    print(f"{name}: shape={df.shape}, cols={list(df.columns)}")
    print(df.head(2))
    if "contribution" in df.columns:
        print("  positive rate:", df["contribution"].mean())


train: shape=(1360850, 3), cols=['timestamp', 'OwnerUserId', 'contribution']
   timestamp  OwnerUserId  contribution
0 2012-01-12          352             1
1 2010-10-14            7             1
  positive rate: 0.04998346621596796
val: shape=(85838, 3), cols=['timestamp', 'OwnerUserId', 'contribution']
   timestamp  OwnerUserId  contribution
0 2020-10-01       246773             1
1 2020-10-01       199619             1
  positive rate: 0.02808779328502528
test: shape=(88137, 2), cols=['timestamp', 'OwnerUserId']
   timestamp  OwnerUserId
0 2021-01-01          668
1 2021-01-01       125849


In [5]:
for name, tbl in db.table_dict.items():
    print(f"{name}: pkey={tbl.pkey_col}, fkeys={tbl.fkey_col_to_pkey_table}")


users: pkey=Id, fkeys={}
postHistory: pkey=Id, fkeys={'PostId': 'posts', 'UserId': 'users'}
postLinks: pkey=Id, fkeys={'PostId': 'posts', 'RelatedPostId': 'posts'}
votes: pkey=Id, fkeys={'PostId': 'posts', 'UserId': 'users'}
posts: pkey=Id, fkeys={'OwnerUserId': 'users', 'ParentId': 'posts'}
badges: pkey=Id, fkeys={'UserId': 'users'}
comments: pkey=Id, fkeys={'UserId': 'users', 'PostId': 'posts'}


## Q1 — Three manual join-aggregate features

In [6]:
import pandas as pd
import numpy as np

users_df    = db.table_dict["users"].df
posts_df    = db.table_dict["posts"].df
votes_df    = db.table_dict["votes"].df
comments_df = db.table_dict["comments"].df

UID = "OwnerUserId"  # user-id column in the target table

def _count_before(target_df, src_df, user_col, name):
    """Count rows of src_df per (user, timestamp) where src CreationDate <= timestamp."""
    s = src_df[[user_col, "CreationDate"]].dropna(subset=[user_col]).copy()
    s[user_col] = s[user_col].astype("int64")
    s = s.rename(columns={user_col: UID})
    m = target_df[[UID, "timestamp"]].merge(s, on=UID, how="left")
    valid = m["CreationDate"].notna() & (m["CreationDate"] <= m["timestamp"])
    return (m.assign(valid=valid)
              .groupby([UID, "timestamp"])["valid"].sum()
              .astype("int64").rename(name))

def build_q1_features(target_df):
    cols = [UID, "timestamp"] + (["contribution"] if "contribution" in target_df.columns else [])
    out = target_df[cols].copy()

    # Feature 1: # posts owned by user before timestamp
    f1 = _count_before(out, posts_df, "OwnerUserId", "num_posts_before_t")

    # Feature 2: # votes cast by user before timestamp
    f2 = _count_before(out, votes_df, "UserId", "num_votes_before_t")

    # Feature 3: days since user's last comment (NaN if user never commented before t)
    c = comments_df[["UserId", "CreationDate"]].dropna(subset=["UserId"]).copy()
    c["UserId"] = c["UserId"].astype("int64")
    c = c.rename(columns={"UserId": UID})
    m = out[[UID, "timestamp"]].merge(c, on=UID, how="left")
    m = m[m["CreationDate"].notna() & (m["CreationDate"] <= m["timestamp"])]
    last_c = m.groupby([UID, "timestamp"])["CreationDate"].max().rename("last_comment_t")
    joined = out.set_index([UID, "timestamp"]).join(last_c)
    delta = joined.index.get_level_values("timestamp") - joined["last_comment_t"]
    f3 = (delta.dt.total_seconds() / 86400.0).rename("days_since_last_comment")

    out = out.set_index([UID, "timestamp"]).join([f1, f2, f3]).reset_index()
    return out

train_q1 = build_q1_features(train_table.df)
print("shape:", train_q1.shape)
print(train_q1.head())
print("\nNaN counts:\n", train_q1.isna().sum())
print("\nDescribe:\n", train_q1[["num_posts_before_t", "num_votes_before_t", "days_since_last_comment"]].describe())

shape: (1360850, 6)
   OwnerUserId  timestamp  contribution  num_posts_before_t  \
0          352 2012-01-12             1                 218   
1            7 2010-10-14             1                  80   
2         1693 2011-01-13             1                   6   
3         1160 2011-04-14             1                   7   
4         1895 2011-04-14             1                  31   

   num_votes_before_t  days_since_last_comment  
0                   2                 2.231586  
1                   0                 0.621124  
2                   0                 6.150768  
3                   0               157.357496  
4                   0                 0.272961  

NaN counts:
 OwnerUserId                     0
timestamp                       0
contribution                    0
num_posts_before_t              0
num_votes_before_t              0
days_since_last_comment    545289
dtype: int64

Describe:
        num_posts_before_t  num_votes_before_t  days_since_last_c

## Q2 — Automatic join-aggregate feature generation

In [7]:
from collections import defaultdict
import pandas as pd
import numpy as np

# ---------- schema graph ----------
def build_schema_graph(db):
    """Each edge: (neighbor_table, col_in_curr, col_in_neighbor, edge_label)."""
    schema = db.table_dict
    adj = defaultdict(list)
    for tname, tbl in schema.items():
        for fk_col, ref_table in tbl.fkey_col_to_pkey_table.items():
            ref_pk = schema[ref_table].pkey_col
            # forward edge: tname --(fk_col -> ref_pk)--> ref_table
            adj[tname].append((ref_table, fk_col, ref_pk, f"{tname}.{fk_col}->{ref_table}.{ref_pk}"))
            # reverse edge: ref_table --(ref_pk -> fk_col)--> tname
            adj[ref_table].append((tname, ref_pk, fk_col, f"{ref_table}.{ref_pk}->{tname}.{fk_col}"))
    return adj

def enumerate_paths(adj, start, max_depth):
    """All simple paths starting at `start` of length 1..max_depth.
    Each path is a list of edges (neighbor, col_in_prev, col_in_neighbor, label)."""
    out = []
    def dfs(curr, visited, edges):
        if len(edges) >= 1:
            out.append(list(edges))
        if len(edges) >= max_depth:
            return
        for (nxt, lcol, rcol, lbl) in adj[curr]:
            if nxt in visited: 
                continue
            visited.add(nxt)
            edges.append((nxt, lcol, rcol, lbl))
            dfs(nxt, visited, edges)
            edges.pop()
            visited.remove(nxt)
    dfs(start, {start}, [])
    return out

# ---------- per-table info: which column is the "creation time", and which cols are aggregable ----------
TIME_COL = {
    "users": "CreationDate", "posts": "CreationDate", "postHistory": "CreationDate",
    "postLinks": "CreationDate", "votes": "CreationDate", "comments": "CreationDate",
    "badges": "Date",
}

def aggregable_columns(db, table_name):
    """Return list of (col_name, kind) for columns we will aggregate.
    Per Piazza: skip PK + FK columns, drop users.AccountId entirely.
    Per PDF schema: badges.Class and badges.TagBased are categorical (not numeric)."""
    tbl = db.table_dict[table_name]
    skip = {tbl.pkey_col} | set(tbl.fkey_col_to_pkey_table.keys())
    if table_name == "users":
        skip.add("AccountId")  # Piazza @10: drop entirely

    # PDF schema labels these as Categorical even though dtype is numeric/bool
    explicit_categorical = {
        ("badges", "Class"),
        ("badges", "TagBased"),
    }

    out = []
    for c, dtype in tbl.df.dtypes.items():
        if c in skip:
            continue
        if (table_name, c) in explicit_categorical:
            out.append((c, "text"))  # categorical = count only
        elif pd.api.types.is_datetime64_any_dtype(dtype):
            out.append((c, "time"))
        elif pd.api.types.is_numeric_dtype(dtype) or pd.api.types.is_bool_dtype(dtype):
            out.append((c, "num"))
        else:
            out.append((c, "text"))
    return out

def count_features_for_depth(db, max_depth):
    """Count NEW features per depth.
    Depth-1 (target ⟕ users) includes raw users columns AND aggregations on them.
    Depth-2+ adds aggregations only from leaf tables along paths from users."""
    adj = build_schema_graph(db)

    n = 0
    # depth-1: raw users columns (excluding AccountId per Piazza) + aggregations
    users_tbl = db.table_dict["users"]
    raw_users_cols = [c for c in users_tbl.df.columns if c != "AccountId"]
    n += len(raw_users_cols)  # raw columns kept from join

    cols = aggregable_columns(db, "users")
    n += sum(3 if k == "num" else (1 if k == "text" else 2) for _, k in cols)

    if max_depth >= 2:
        paths = enumerate_paths(adj, "users", max_depth - 1)
        for edges in paths:
            leaf = edges[-1][0]
            cols = aggregable_columns(db, leaf)
            n += sum(3 if k == "num" else (1 if k == "text" else 2) for _, k in cols)
    return n

# sanity check
adj = build_schema_graph(db)
print("schema adjacency:")
for k, vs in adj.items():
    print(" ", k, "->", [(v[0], v[3]) for v in vs])

print("\nQ3: feature counts by depth")
for d in [1, 2, 3]:
    print(f"  max_depth={d}: {count_features_for_depth(db, d)} features")

schema adjacency:
  postHistory -> [('posts', 'postHistory.PostId->posts.Id'), ('users', 'postHistory.UserId->users.Id')]
  posts -> [('postHistory', 'posts.Id->postHistory.PostId'), ('postLinks', 'posts.Id->postLinks.PostId'), ('postLinks', 'posts.Id->postLinks.RelatedPostId'), ('votes', 'posts.Id->votes.PostId'), ('users', 'posts.OwnerUserId->users.Id'), ('posts', 'posts.ParentId->posts.Id'), ('posts', 'posts.Id->posts.ParentId'), ('comments', 'posts.Id->comments.PostId')]
  users -> [('postHistory', 'users.Id->postHistory.UserId'), ('votes', 'users.Id->votes.UserId'), ('posts', 'users.Id->posts.OwnerUserId'), ('badges', 'users.Id->badges.UserId'), ('comments', 'users.Id->comments.UserId')]
  postLinks -> [('posts', 'postLinks.PostId->posts.Id'), ('posts', 'postLinks.RelatedPostId->posts.Id')]
  votes -> [('posts', 'votes.PostId->posts.Id'), ('users', 'votes.UserId->users.Id')]
  badges -> [('users', 'badges.UserId->users.Id')]
  comments -> [('users', 'comments.UserId->users.Id'), (

In [8]:
# ----- Q2: materialize the feature_matrix at max_depth=2 -----
import time

UID = "OwnerUserId"

def _agg_dict(cols_info):
    d = {}
    for c, kind in cols_info:
        if kind == "num":
            d[c] = ["mean", "count", "sum"]
        elif kind == "text":
            d[c] = ["count"]
        else:  # time
            d[c] = ["min", "max"]
    return d

def build_feature_matrix(target_df, db, max_depth=2, verbose=True):
    """Build feature matrix with:
       - depth-1: target ⟕ users; keep raw users columns (except AccountId) AND emit aggregations.
       - depth-2: aggregate columns of each user-neighbor table (skip PK/FK per Piazza).
       - temporal cutoff: each joined row must have creation_date <= target.timestamp.
    """
    assert max_depth == 2
    t0 = time.time()
    base_cols = [UID, "timestamp"] + (["contribution"] if "contribution" in target_df.columns else [])
    fm = target_df[base_cols].reset_index(drop=True).copy()
    fm["_tid"] = np.arange(len(fm), dtype=np.int64)

    # ---------- depth-1: target ⟕ users (KEEP raw columns + emit aggregations) ----------
    if verbose: print("[depth-1] target ⟕ users ...")
    users_df = db.table_dict["users"].df.copy()
    users_df["Id"] = users_df["Id"].astype("int64")
    # drop AccountId per Piazza @10
    users_df = users_df.drop(columns=["AccountId"])
    j = fm[["_tid", UID]].merge(users_df, left_on=UID, right_on="Id", how="left").set_index("_tid")

    # Keep raw users columns (rename with users__ prefix)
    raw_users = j.drop(columns=[UID]).copy()
    raw_users.columns = [f"users__{c}" for c in raw_users.columns]

    # Aggregations (per Piazza: skip Id PK and AccountId already dropped above)
    new_cols = {}
    for col, kind in aggregable_columns(db, "users"):
        if kind == "num":
            new_cols[f"users__{col}__mean"]  = j[col].astype("float64")
            new_cols[f"users__{col}__count"] = j[col].notna().astype("int64")
            new_cols[f"users__{col}__sum"]   = j[col].astype("float64")
        elif kind == "text":
            new_cols[f"users__{col}__count"] = j[col].notna().astype("int64")
        else:  # time
            new_cols[f"users__{col}__min"]   = j[col]
            new_cols[f"users__{col}__max"]   = j[col]
    d1 = pd.DataFrame(new_cols)

    # ---------- depth-2: target ⟕ users ⋈ X ----------
    depth2 = [
        ("posts",       "OwnerUserId"),
        ("postHistory", "UserId"),
        ("votes",       "UserId"),
        ("badges",      "UserId"),
        ("comments",    "UserId"),
    ]
    depth2_frames = []
    for tname, user_fk in depth2:
        if verbose: print(f"[depth-2] users ⋈ {tname} (on {user_fk}) ...", end=" ")
        time_col = TIME_COL[tname]
        cols_info = aggregable_columns(db, tname)
        keep = list(dict.fromkeys([user_fk, time_col] + [c for c, _ in cols_info]))
        src = db.table_dict[tname].df[keep].copy()
        src = src.dropna(subset=[user_fk])
        src[user_fk] = src[user_fk].astype("int64")

        m = fm[["_tid", UID, "timestamp"]].merge(src, left_on=UID, right_on=user_fk, how="inner")
        m = m[m[time_col] <= m["timestamp"]]
        if verbose: print(f"matched rows: {len(m):,}")
        if len(m) == 0:
            continue

        g = m.groupby("_tid").agg(_agg_dict(cols_info))
        g.columns = [f"{tname}__{c}__{a}" for c, a in g.columns]
        depth2_frames.append(g)

    # ---------- assemble ----------
    feature_matrix = (
        fm.set_index("_tid")
          .join(raw_users)
          .join(d1)
          .join(depth2_frames, how="left")
    )
    count_cols = [c for c in feature_matrix.columns if c.endswith("__count")]
    feature_matrix[count_cols] = feature_matrix[count_cols].fillna(0).astype("int64")
    feature_matrix = feature_matrix.reset_index(drop=True)
    if verbose: print(f"feature_matrix shape: {feature_matrix.shape}   ({time.time()-t0:.1f}s)")
    return feature_matrix

fm_train = build_feature_matrix(train_table.df, db, max_depth=2)
print("\ncolumns ({}):".format(len(fm_train.columns)))
print(list(fm_train.columns))
fm_train.head(2)


[depth-1] target ⟕ users ...
[depth-2] users ⋈ posts (on OwnerUserId) ... matched rows: 5,250,955
[depth-2] users ⋈ postHistory (on UserId) ... matched rows: 16,814,778
[depth-2] users ⋈ votes (on UserId) ... matched rows: 77,552
[depth-2] users ⋈ badges (on UserId) ... matched rows: 5,425,886
[depth-2] users ⋈ comments (on UserId) ... matched rows: 9,705,158
feature_matrix shape: (1360850, 54)   (41.3s)

columns (54):
['OwnerUserId', 'timestamp', 'contribution', 'users__Id', 'users__DisplayName', 'users__Location', 'users__ProfileImageUrl', 'users__WebsiteUrl', 'users__AboutMe', 'users__CreationDate', 'users__DisplayName__count', 'users__Location__count', 'users__ProfileImageUrl__mean', 'users__ProfileImageUrl__count', 'users__ProfileImageUrl__sum', 'users__WebsiteUrl__count', 'users__AboutMe__count', 'users__CreationDate__min', 'users__CreationDate__max', 'posts__PostTypeId__mean', 'posts__PostTypeId__count', 'posts__PostTypeId__sum', 'posts__OwnerDisplayName__count', 'posts__Title__

,OwnerUserId,timestamp,contribution,users__Id,users__DisplayName,users__Location,users__ProfileImageUrl,users__WebsiteUrl,users__AboutMe,users__CreationDate,...,badges__Class__count,badges__Name__count,badges__TagBased__count,badges__Date__min,badges__Date__max,comments__ContentLicense__count,comments__UserDisplayName__count,comments__Text__count,comments__CreationDate__min,comments__CreationDate__max
0,352,2012-01-12,1,352,onestop,United Kingdom,NaN,http://www.onestop.co.uk,<p>Statistician currently based in the UK.</p>\n,2010-07-27 12:10:34.843,...,48,48,48,2010-07-27 12:18:44.327,2012-01-08 18:31:28.823,353,0,353,2010-08-20 12:42:10.483,2012-01-09 18:26:30.950
1,7,2010-10-14,1,7,csgillespie,"Newcastle, United Kingdom",NaN,https://www.jumpingrivers.com/,<p>I'm a senior statistics lecturer at Newcast...,2010-07-19 19:04:52.280,...,25,25,25,2010-07-19 19:39:07.330,2010-10-13 16:59:17.470,72,0,72,2010-07-19 19:56:18.490,2010-10-13 09:05:34.927


In [11]:
# Build feature matrices for val/test too, and persist all three to disk.
import os
os.makedirs("features", exist_ok=True)

fm_val  = build_feature_matrix(val_table.df,  db, max_depth=2)
fm_test = build_feature_matrix(test_table.df, db, max_depth=2)

fm_train.to_pickle("features/fm_train.pkl")
fm_val.to_pickle("features/fm_val.pkl")
fm_test.to_pickle("features/fm_test.pkl")

print(f"\nsaved:")
print(f"  fm_train: {fm_train.shape}")
print(f"  fm_val:   {fm_val.shape}")
print(f"  fm_test:  {fm_test.shape}")

[depth-1] target ⟕ users ...
[depth-2] users ⋈ posts (on OwnerUserId) ... matched rows: 319,820
[depth-2] users ⋈ postHistory (on UserId) ... matched rows: 1,066,932
[depth-2] users ⋈ votes (on UserId) ... matched rows: 5,017
[depth-2] users ⋈ badges (on UserId) ... matched rows: 367,072
[depth-2] users ⋈ comments (on UserId) ... matched rows: 594,598
feature_matrix shape: (85838, 54)   (3.1s)
[depth-1] target ⟕ users ...
[depth-2] users ⋈ posts (on OwnerUserId) ... matched rows: 328,648
[depth-2] users ⋈ postHistory (on UserId) ... matched rows: 1,098,857
[depth-2] users ⋈ votes (on UserId) ... matched rows: 5,182
[depth-2] users ⋈ badges (on UserId) ... matched rows: 380,489
[depth-2] users ⋈ comments (on UserId) ... matched rows: 612,288
feature_matrix shape: (88137, 53)   (2.8s)

saved:
  fm_train: (1360850, 54)
  fm_val:   (85838, 54)
  fm_test:  (88137, 53)


## Q3–Q6 — Feature counts, shape, examples, missing values

## Q7 — Train and compare four classifiers

In [16]:
# ---------- Preprocessing for Q7 ----------
import numpy as np
import pandas as pd

def prepare_xy(fm, drop_cols=("OwnerUserId", "timestamp", "contribution")):
    """Return (X DataFrame with numeric features only, y array or None)."""
    y = fm["contribution"].values.astype("int64") if "contribution" in fm.columns else None
    X = fm.drop(columns=[c for c in drop_cols if c in fm.columns]).copy()
    for c in X.columns:
        if pd.api.types.is_datetime64_any_dtype(X[c]):
            # convert to seconds since epoch (smaller magnitude than ns) to be safe in float32
            X[c] = X[c].astype("int64").astype("float64") / 1e9
            X[c] = X[c].mask(~np.isfinite(X[c]), -1)
        elif X[c].dtype == "object":
            X[c] = pd.to_numeric(X[c], errors="coerce")
        else:
            X[c] = X[c].astype("float64")
    # Final cleanup: replace inf with NaN; clip to safe float32 range
    X = X.replace([np.inf, -np.inf], np.nan)
    return X, y

X_train, y_train = prepare_xy(fm_train)
X_val,   y_val   = prepare_xy(fm_val)

print("X_train:", X_train.shape, " y_train pos-rate:", y_train.mean().round(4))
print("X_val:  ", X_val.shape,   " y_val pos-rate:  ", y_val.mean().round(4))
print("dtypes:", X_train.dtypes.value_counts().to_dict())
print("nan share per col (top 5):")
print(X_train.isna().mean().sort_values(ascending=False).head(5))
# quick sanity check for any remaining inf
finite_check = np.isfinite(X_train.fillna(0).values).all()
print("all finite after fillna(0):", finite_check)
max_abs = X_train.fillna(0).abs().max().max()
print(f"max abs value: {max_abs:.3e}")

X_train: (1360850, 51)  y_train pos-rate: 0.05
X_val:   (85838, 51)  y_val pos-rate:   0.0281
dtypes: {dtype('float64'): 51}
nan share per col (top 5):
users__AboutMe                  1.0
users__WebsiteUrl               1.0
users__ProfileImageUrl          1.0
users__ProfileImageUrl__sum     1.0
users__ProfileImageUrl__mean    1.0
dtype: float64
all finite after fillna(0): True
max abs value: 1.010e+11


In [17]:
# ---------- Eval harness ----------
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
)

def eval_split(y_true, y_pred, y_score):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "roc_auc":   roc_auc_score(y_true, y_score),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }

def fmt(metrics_tr, metrics_val):
    return {k: f"{metrics_tr[k]:.3f}/{metrics_val[k]:.3f}" for k in metrics_tr}

# Big dict to collect every variant we try, for the per-model variant tables in the report.
all_runs = []
def log_run(model_name, variant, m_tr, m_val):
    all_runs.append({"model": model_name, "variant": variant,
                     **{f"tr_{k}": v for k, v in m_tr.items()},
                     **{f"val_{k}": v for k, v in m_val.items()}})

In [18]:
# ---------- Decision Tree ----------
from sklearn.tree import DecisionTreeClassifier

# DT in sklearn < 1.3 does NOT handle NaN. Fill with -1 sentinel.
X_train_dt = X_train.fillna(-1).values
X_val_dt   = X_val.fillna(-1).values

dt_grid = [
    {"max_depth": 5},
    {"max_depth": 10},
    {"max_depth": 20},
    {"max_depth": None, "min_samples_leaf": 50},
]
best_dt, best_dt_val_auc = None, -1
for params in dt_grid:
    clf = DecisionTreeClassifier(class_weight="balanced", random_state=0, **params)
    clf.fit(X_train_dt, y_train)
    s_tr  = clf.predict_proba(X_train_dt)[:, 1]
    s_val = clf.predict_proba(X_val_dt)[:, 1]
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("DecisionTree", str(params), m_tr, m_val)
    print(f"DT {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_dt_val_auc:
        best_dt, best_dt_val_auc = clf, m_val["roc_auc"]
        best_dt_params, best_dt_m_tr, best_dt_m_val = params, m_tr, m_val

print("\nBEST DT:", best_dt_params, "val AUC =", round(best_dt_val_auc, 3))

DT {'max_depth': 5}: val AUC=0.861, val F1=0.130
DT {'max_depth': 10}: val AUC=0.839, val F1=0.144
DT {'max_depth': 20}: val AUC=0.606, val F1=0.177
DT {'max_depth': None, 'min_samples_leaf': 50}: val AUC=0.840, val F1=0.150

BEST DT: {'max_depth': 5} val AUC = 0.861


In [19]:
# ---------- XGBoost ----------
from xgboost import XGBClassifier

pos_ratio = float((y_train == 0).sum() / (y_train == 1).sum())  # ~19 for ~5% positive

xgb_grid = [
    {"max_depth": 4,  "n_estimators": 200, "learning_rate": 0.10},
    {"max_depth": 6,  "n_estimators": 200, "learning_rate": 0.10},
    {"max_depth": 8,  "n_estimators": 400, "learning_rate": 0.05},
]
best_xgb, best_xgb_val_auc = None, -1
for params in xgb_grid:
    clf = XGBClassifier(
        scale_pos_weight=pos_ratio,
        tree_method="hist", n_jobs=-1, random_state=0,
        eval_metric="logloss", **params,
    )
    clf.fit(X_train.values, y_train)
    s_tr  = clf.predict_proba(X_train.values)[:, 1]
    s_val = clf.predict_proba(X_val.values)[:, 1]
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("XGBoost", str(params), m_tr, m_val)
    print(f"XGB {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_xgb_val_auc:
        best_xgb, best_xgb_val_auc = clf, m_val["roc_auc"]
        best_xgb_params, best_xgb_m_tr, best_xgb_m_val = params, m_tr, m_val

print("\nBEST XGB:", best_xgb_params, "val AUC =", round(best_xgb_val_auc, 3))

XGB {'max_depth': 4, 'n_estimators': 200, 'learning_rate': 0.1}: val AUC=0.879, val F1=0.153
XGB {'max_depth': 6, 'n_estimators': 200, 'learning_rate': 0.1}: val AUC=0.877, val F1=0.156
XGB {'max_depth': 8, 'n_estimators': 400, 'learning_rate': 0.05}: val AUC=0.867, val F1=0.167

BEST XGB: {'max_depth': 4, 'n_estimators': 200, 'learning_rate': 0.1} val AUC = 0.879


In [20]:
# ---------- Logistic Regression ----------
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy="median").fit(X_train.values)
X_tr_imp = imp.transform(X_train.values)
X_va_imp = imp.transform(X_val.values)
scaler = StandardScaler().fit(X_tr_imp)
X_tr_sc = scaler.transform(X_tr_imp)
X_va_sc = scaler.transform(X_va_imp)

lr_grid = [
    {"C": 0.1},
    {"C": 1.0},
    {"C": 10.0},
]
best_lr, best_lr_val_auc = None, -1
for params in lr_grid:
    clf = LogisticRegression(
        class_weight="balanced", max_iter=2000, solver="lbfgs",
        n_jobs=-1, random_state=0, **params,
    )
    clf.fit(X_tr_sc, y_train)
    s_tr  = clf.predict_proba(X_tr_sc)[:, 1]
    s_val = clf.predict_proba(X_va_sc)[:, 1]
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("LogReg", str(params), m_tr, m_val)
    print(f"LR {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_lr_val_auc:
        best_lr, best_lr_val_auc = clf, m_val["roc_auc"]
        best_lr_params, best_lr_m_tr, best_lr_m_val = params, m_tr, m_val

print("\nBEST LR:", best_lr_params, "val AUC =", round(best_lr_val_auc, 3))

/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 3  4  5  9 11]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 3  4  5  9 11]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


LR {'C': 0.1}: val AUC=0.839, val F1=0.138
LR {'C': 1.0}: val AUC=0.830, val F1=0.140
LR {'C': 10.0}: val AUC=0.828, val F1=0.140

BEST LR: {'C': 0.1} val AUC = 0.839


### Custom NN (MLP) — architecture

In [21]:
# ---------- Custom NN (PyTorch MLP) ----------
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

# reuse imputed+scaled features from the LR cell (X_tr_sc, X_va_sc)
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(128, 64), dropout=0.2):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_mlp(hidden, dropout, lr, epochs=4, batch_size=4096):
    torch.manual_seed(0)
    Xt = torch.tensor(X_tr_sc, dtype=torch.float32)
    yt = torch.tensor(y_train, dtype=torch.float32)
    Xv = torch.tensor(X_va_sc, dtype=torch.float32).to(device)
    yv = torch.tensor(y_val,   dtype=torch.float32).to(device)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    model = MLP(Xt.shape[1], hidden=hidden, dropout=dropout).to(device)
    pos_w = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)], dtype=torch.float32, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            s_val = torch.sigmoid(model(Xv)).cpu().numpy()
        print(f"  epoch {ep+1}: val AUC={roc_auc_score(y_val, s_val):.3f}")
    return model

def score_mlp(model, X_np):
    model.eval()
    with torch.no_grad():
        scores = []
        for i in range(0, len(X_np), 65536):
            xb = torch.tensor(X_np[i:i+65536], dtype=torch.float32, device=device)
            scores.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(scores)

nn_grid = [
    {"hidden": (128, 64), "dropout": 0.2, "lr": 1e-3, "epochs": 4},
    {"hidden": (256, 128, 64), "dropout": 0.3, "lr": 1e-3, "epochs": 4},
]
best_nn, best_nn_val_auc = None, -1
for params in nn_grid:
    print("NN", params)
    model = train_mlp(**params)
    s_tr  = score_mlp(model, X_tr_sc)
    s_val = score_mlp(model, X_va_sc)
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("MLP", str(params), m_tr, m_val)
    print(f"  -> val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}\n")
    if m_val["roc_auc"] > best_nn_val_auc:
        best_nn, best_nn_val_auc = model, m_val["roc_auc"]
        best_nn_params, best_nn_m_tr, best_nn_m_val = params, m_tr, m_val

print("BEST NN:", best_nn_params, "val AUC =", round(best_nn_val_auc, 3))

using device: cuda
NN {'hidden': (128, 64), 'dropout': 0.2, 'lr': 0.001, 'epochs': 4}
  epoch 1: val AUC=0.852
  epoch 2: val AUC=0.856
  epoch 3: val AUC=0.856
  epoch 4: val AUC=0.854
  -> val AUC=0.854, val F1=0.144

NN {'hidden': (256, 128, 64), 'dropout': 0.3, 'lr': 0.001, 'epochs': 4}
  epoch 1: val AUC=0.851
  epoch 2: val AUC=0.854
  epoch 3: val AUC=0.857
  epoch 4: val AUC=0.851
  -> val AUC=0.851, val F1=0.143

BEST NN: {'hidden': (128, 64), 'dropout': 0.2, 'lr': 0.001, 'epochs': 4} val AUC = 0.854


In [22]:
# ---------- Final summary tables ----------
summary_rows = [
    {"model": "Decision Tree",       **fmt(best_dt_m_tr,  best_dt_m_val)},
    {"model": "XGBoost",             **fmt(best_xgb_m_tr, best_xgb_m_val)},
    {"model": "Logistic Regression", **fmt(best_lr_m_tr,  best_lr_m_val)},
    {"model": "Custom NN (MLP)",     **fmt(best_nn_m_tr,  best_nn_m_val)},
]
summary = pd.DataFrame(summary_rows).set_index("model")
print("=== Q7 best-variant summary (train/val) ===")
print(summary)

print("\n=== Q7 full hyperparam grid (every variant tried) ===")
runs_df = pd.DataFrame(all_runs)
# round numeric cols
for c in runs_df.columns:
    if pd.api.types.is_numeric_dtype(runs_df[c]):
        runs_df[c] = runs_df[c].round(3)
print(runs_df.to_string(index=False))

# persist for the report
summary.to_csv("features/q7_summary.csv")
runs_df.to_csv("features/q7_all_runs.csv", index=False)

=== Q7 best-variant summary (train/val) ===
                        accuracy      roc_auc    precision       recall  \
model                                                                     
Decision Tree        0.769/0.670  0.808/0.861  0.141/0.070  0.708/0.880   
XGBoost              0.790/0.735  0.846/0.879  0.158/0.084  0.736/0.852   
Logistic Regression  0.778/0.718  0.798/0.839  0.140/0.075  0.669/0.802   
Custom NN (MLP)      0.785/0.733  0.814/0.854  0.146/0.079  0.682/0.800   

                              f1  
model                             
Decision Tree        0.235/0.130  
XGBoost              0.260/0.153  
Logistic Regression  0.232/0.138  
Custom NN (MLP)      0.241/0.144  

=== Q7 full hyperparam grid (every variant tried) ===
       model                                                              variant  tr_accuracy  tr_roc_auc  tr_precision  tr_recall  tr_f1  val_accuracy  val_roc_auc  val_precision  val_recall  val_f1
DecisionTree                           

### Q7 — Final results table

## Q8–Q9 — Metric definitions and results discussion

## Q10–Q11 — Feature importance: method + top-10 per model

In [23]:
# ---------- Q10/Q11: feature importances ----------
feature_names = list(X_train.columns)

def top_k(importances, k=10):
    idx = np.argsort(importances)[::-1][:k]
    return pd.DataFrame({"feature": [feature_names[i] for i in idx],
                         "importance": importances[idx]})

dt_top  = top_k(best_dt.feature_importances_)
xgb_top = top_k(best_xgb.feature_importances_)
lr_top  = top_k(np.abs(best_lr.coef_[0]))

print("=== DT top-10 ===");  print(dt_top.to_string(index=False))
print("\n=== XGB top-10 ==="); print(xgb_top.to_string(index=False))
print("\n=== LR top-10 (|coef| on standardized features) ==="); print(lr_top.to_string(index=False))

# Save side-by-side for the report
top10 = pd.concat({
    "Decision Tree": dt_top.reset_index(drop=True),
    "XGBoost":       xgb_top.reset_index(drop=True),
    "LogReg":        lr_top.reset_index(drop=True),
}, axis=1)
top10.to_csv("features/q11_top10.csv")

=== DT top-10 ===
                       feature  importance
            posts__Body__count    0.697928
      posts__PostTypeId__count    0.132071
         users__AboutMe__count    0.045286
             badges__Date__max    0.031734
      posts__CreationDate__max    0.027581
   comments__CreationDate__max    0.026463
postHistory__CreationDate__max    0.022671
         comments__Text__count    0.009612
           badges__Name__count    0.003348
  posts__ContentLicense__count    0.001766

=== XGB top-10 ===
                            feature  importance
           posts__PostTypeId__count    0.519744
             posts__PostTypeId__sum    0.148867
              users__AboutMe__count    0.043094
    comments__ContentLicense__count    0.042839
             users__Location__count    0.035225
postHistory__PostHistoryTypeId__sum    0.028623
     postHistory__CreationDate__max    0.025488
        comments__CreationDate__max    0.022977
                  badges__Date__max    0.021013
         

### Q11 — Top-10 discussion

## Part 2 — Q12: Graph construction and structural features

In [1]:
import pandas as pd, numpy as np, networkx as nx, time, gc
from collections import defaultdict
from relbench.datasets import get_dataset
from relbench.tasks import get_task

dataset = get_dataset("rel-stack", download=True)
db = dataset.get_db()
task = get_task("rel-stack", "user-engagement", download=True)
train_table = task.get_table("train")
val_table   = task.get_table("val")
test_table  = task.get_table("test")

fm_train = pd.read_pickle("features/fm_train.pkl")
fm_val   = pd.read_pickle("features/fm_val.pkl")
fm_test  = pd.read_pickle("features/fm_test.pkl")

print(fm_train.shape)  # should be (1360850, 99)


/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Database object from /home/abed/.cache/relbench/rel-stack/db...
Done in 12.13 seconds.
(1360850, 54)


In [2]:
# ----- Q12: helpers for temporal graph + features -----
import networkx as nx
import time, gc

VAL_TIMESTAMP = pd.Timestamp("2020-10-01 00:00:00")
TEST_TIMESTAMP = pd.Timestamp("2021-01-01 00:00:00")

# Time-column lookup per table (the column we filter on for temporal cutoffs)
TIME_COL = {
    "users": "CreationDate", "posts": "CreationDate", "postHistory": "CreationDate",
    "postLinks": "CreationDate", "votes": "CreationDate", "comments": "CreationDate",
    "badges": "Date",
}

def get_valid_ids(db, time_cutoff=None, inclusive=False):
    """For each table, return the set of PK ids of tuples whose timestamp passes the filter."""
    valid_ids = {}
    for tname, tbl in db.table_dict.items():
        time_col = TIME_COL[tname]
        df = tbl.df
        if time_cutoff is None:
            mask = pd.Series(True, index=df.index)
        elif inclusive:
            mask = df[time_col] <= time_cutoff
        else:
            mask = df[time_col] < time_cutoff
        valid_ids[tname] = set(df.loc[mask, tbl.pkey_col].dropna().astype("int64"))
    return valid_ids

def build_temporal_graph(db, valid_ids, verbose=True):
    """Build a networkx graph with nodes = valid tuples and edges = FK-PK connections
    between valid tuples."""
    t0 = time.time()
    G = nx.Graph()
    # add all valid nodes (so isolated ones get degree 0, not missing)
    for tname, ids in valid_ids.items():
        G.add_nodes_from(((tname, int(i)) for i in ids))
    # add edges
    n_edges = 0
    for tname, tbl in db.table_dict.items():
        pk = tbl.pkey_col
        for fk_col, ref_table in tbl.fkey_col_to_pkey_table.items():
            sub = tbl.df[[pk, fk_col]].dropna(subset=[fk_col]).copy()
            sub[fk_col] = sub[fk_col].astype("int64")
            sub[pk] = sub[pk].astype("int64")
            mask_src = sub[pk].isin(valid_ids[tname])
            mask_dst = sub[fk_col].isin(valid_ids[ref_table])
            sub = sub[mask_src & mask_dst]
            edges = list(zip(
                zip([tname] * len(sub), sub[pk].tolist()),
                zip([ref_table] * len(sub), sub[fk_col].tolist()),
            ))
            G.add_edges_from(edges)
            n_edges += len(sub)
    if verbose:
        print(f"  graph built: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges ({time.time()-t0:.1f}s)")
    return G

def wl_user_colors(G, user_nodes, K_list=(2, 3, 5)):
    """Run WL hashing on G for max(K_list) iterations, snapshot user-node colors at each K."""
    t0 = time.time()
    color = {n: hash(n[0]) for n in G.nodes()}  # initial: table name hash
    user_set = set(user_nodes)
    adj = G.adj
    K_set = set(K_list)
    snaps = {}
    for it in range(1, max(K_list) + 1):
        new_color = {}
        for n in color:
            nbr = adj[n]
            if nbr:
                nbr_colors = sorted(color[m] for m in nbr)
                new_color[n] = hash((color[n], tuple(nbr_colors)))
            else:
                new_color[n] = hash((color[n], ()))
        if it in K_set:
            snaps[it] = {u: new_color[u] for u in user_set if u in new_color}
        color = new_color
    print(f"  WL done ({time.time()-t0:.1f}s)")
    return snaps

def graphlet_counts(db, valid_ids):
    """Compute the two graphlet counts using pandas joins, filtered by valid_ids."""
    # Graphlet 1: user-comment-post-user (user commented on own post)
    comments = db.table_dict["comments"].df[["Id", "UserId", "PostId"]].dropna().copy()
    comments["UserId"] = comments["UserId"].astype("int64")
    comments["PostId"] = comments["PostId"].astype("int64")
    comments["Id"] = comments["Id"].astype("int64")
    comments = comments[comments["Id"].isin(valid_ids["comments"])
                        & comments["UserId"].isin(valid_ids["users"])
                        & comments["PostId"].isin(valid_ids["posts"])]

    posts_owner = db.table_dict["posts"].df[["Id", "OwnerUserId"]].dropna(subset=["OwnerUserId"]).copy()
    posts_owner["Id"] = posts_owner["Id"].astype("int64")
    posts_owner["OwnerUserId"] = posts_owner["OwnerUserId"].astype("int64")
    posts_owner = posts_owner[posts_owner["Id"].isin(valid_ids["posts"])
                              & posts_owner["OwnerUserId"].isin(valid_ids["users"])]
    posts_owner = posts_owner.rename(columns={"Id": "PostId", "OwnerUserId": "PostOwnerId"})

    m = comments.merge(posts_owner, on="PostId", how="inner")
    self_comments = m[m["UserId"] == m["PostOwnerId"]]
    gl1 = self_comments.groupby("UserId").size().rename("self_comment_count")

    # Graphlet 2: user-post-postLinks-post-user (user has two linked own posts)
    links = db.table_dict["postLinks"].df[["Id", "PostId", "RelatedPostId"]].dropna().copy()
    links["Id"] = links["Id"].astype("int64")
    links["PostId"] = links["PostId"].astype("int64")
    links["RelatedPostId"] = links["RelatedPostId"].astype("int64")
    links = links[links["Id"].isin(valid_ids["postLinks"])
                  & links["PostId"].isin(valid_ids["posts"])
                  & links["RelatedPostId"].isin(valid_ids["posts"])]

    A = posts_owner.rename(columns={"PostId": "PostId", "PostOwnerId": "OwnerA"})
    B = posts_owner.rename(columns={"PostId": "RelatedPostId", "PostOwnerId": "OwnerB"})
    m2 = links.merge(A, on="PostId", how="inner").merge(B, on="RelatedPostId", how="inner")
    linked_same = m2[m2["OwnerA"] == m2["OwnerB"]]
    gl2 = linked_same.groupby("OwnerA").size().rename("linked_own_posts_count")
    gl2.index.name = "UserId"
    return gl1, gl2

def compute_split_features(db, time_cutoff, inclusive, label):
    """Build a temporal graph and compute all graph features for users. Returns DataFrame."""
    print(f"\n=== Building graph for split: {label} ===")
    valid_ids = get_valid_ids(db, time_cutoff, inclusive)
    print(f"  valid users: {len(valid_ids['users']):,}")
    G = build_temporal_graph(db, valid_ids)

    # degree
    deg = dict(G.degree())
    user_ids_all = db.table_dict["users"].df["Id"].astype("int64").tolist()
    deg_series = pd.Series(
        {uid: deg.get(("users", int(uid)), 0) for uid in user_ids_all},
        name="graph_degree",
    )
    deg_series.index.name = "OwnerUserId"

    # WL colors at K=2,3,5
    user_nodes = [("users", int(uid)) for uid in user_ids_all if uid in valid_ids["users"]]
    wl_snaps = wl_user_colors(G, user_nodes, K_list=(2, 3, 5))

    # free graph memory before computing graphlets
    del G; gc.collect()

    # graphlets
    gl1, gl2 = graphlet_counts(db, valid_ids)

    # Assemble feature DataFrame
    feats = pd.DataFrame(index=pd.Index(user_ids_all, name="OwnerUserId"))
    feats["graph_degree"] = deg_series
    for K in (2, 3, 5):
        cmap = wl_snaps[K]
        uid_to_color = pd.Series(
            {uid: cmap.get(("users", int(uid))) for uid in user_ids_all},
            name=f"wl_K{K}",
        )
        codes, _ = pd.factorize(uid_to_color, use_na_sentinel=True)
        feats[f"wl_K{K}"] = pd.Series(codes, index=uid_to_color.index)
    feats["self_comment_count"] = gl1
    feats["linked_own_posts_count"] = gl2

    # fill counts with 0, fill WL NaNs with -1
    feats["graph_degree"] = feats["graph_degree"].fillna(0).astype("int64")
    feats["self_comment_count"] = feats["self_comment_count"].fillna(0).astype("int64")
    feats["linked_own_posts_count"] = feats["linked_own_posts_count"].fillna(0).astype("int64")
    for K in (2, 3, 5):
        feats[f"wl_K{K}"] = feats[f"wl_K{K}"].fillna(-1).astype("int64")
    return feats

# Compute for train split (timestamps strictly before VAL_TIMESTAMP)
features_train = compute_split_features(db, VAL_TIMESTAMP, inclusive=False, label="train")
print(features_train.head(3))
print(features_train.describe())



=== Building graph for split: train ===
  valid users: 247,398
  graph built: 4,121,359 nodes, 5,641,899 edges (39.3s)
  WL done (91.6s)
             graph_degree  wl_K2  wl_K3  wl_K5  self_comment_count  \
OwnerUserId                                                          
0                   68018      0      0      0                   0   
1                       3      1      1      1                   0   
2                       6      2      2      2                   0   

             linked_own_posts_count  
OwnerUserId                          
0                                 0  
1                                 0  
2                                 0  
        graph_degree          wl_K2          wl_K3          wl_K5  \
count  255360.000000  255360.000000  255360.000000  255360.000000   
mean        9.538906    6457.255725   10935.010867   13567.193523   
std       250.977285   12872.660128   19493.613343   23555.570655   
min         0.000000      -1.000000      -1.0

In [3]:
# ----- Q12: val split graph features -----
import gc
features_val = compute_split_features(db, VAL_TIMESTAMP, inclusive=True, label="val")
gc.collect()
print(features_val.describe())



=== Building graph for split: val ===
  valid users: 247,398
  graph built: 4,121,747 nodes, 5,642,180 edges (35.2s)
  WL done (89.5s)
        graph_degree          wl_K2          wl_K3         wl_K5  \
count  255360.000000  255360.000000  255360.000000  255360.00000   
mean        9.538918    6457.713252   10935.378313   13567.45674   
std       250.977301   12873.763464   19494.499860   23556.16143   
min         0.000000      -1.000000      -1.000000      -1.00000   
25%         0.000000      18.000000      18.000000      18.00000   
50%         1.000000      51.000000      51.000000      51.00000   
75%         5.000000    4600.250000   14005.000000   19716.25000   
max     68018.000000   52251.000000   71104.000000   82275.00000   

       self_comment_count  linked_own_posts_count  
count       255360.000000           255360.000000  
mean             0.769102                0.013436  
std             14.648751                0.264795  
min              0.000000                0.

In [4]:
# ----- Q12: test split graph features -----
import gc
features_test = compute_split_features(db, time_cutoff=None, inclusive=True, label="test")
gc.collect()
print(features_test.describe())



=== Building graph for split: test ===
  valid users: 255,360
  graph built: 4,247,264 nodes, 5,812,887 edges (34.6s)
  WL done (90.4s)
        graph_degree          wl_K2          wl_K3          wl_K5  \
count  255360.000000  255360.000000  255360.000000  255360.000000   
mean        9.827741    6785.381743   11501.750901   14303.527757   
std       258.061809   13319.795667   20155.871751   24399.369971   
min         0.000000       0.000000       0.000000       0.000000   
25%         0.000000      51.000000      51.000000      51.000000   
50%         1.000000      51.000000      51.000000      51.000000   
75%         6.000000    5327.250000   15679.250000   21887.250000   
max     70436.000000   53583.000000   72885.000000   84479.000000   

       self_comment_count  linked_own_posts_count  
count       255360.000000           255360.000000  
mean             0.792861                0.014063  
std             14.942368                0.281606  
min              0.000000        

In [5]:
# ----- Q12: merge graph features into feature matrices and save -----
import os, gc

os.makedirs("features", exist_ok=True)

def add_graph_features(fm, features):
    out = fm.merge(features, left_on="OwnerUserId", right_index=True, how="left")
    # any users not in valid set get default values
    for c in features.columns:
        if c.startswith("wl_"):
            out[c] = out[c].fillna(-1).astype("int64")
        else:
            out[c] = out[c].fillna(0).astype("int64")
    return out

fm_train_g = add_graph_features(fm_train, features_train)
fm_val_g   = add_graph_features(fm_val,   features_val)
fm_test_g  = add_graph_features(fm_test,  features_test)

fm_train_g.to_pickle("features/fm_train_graph.pkl")
fm_val_g.to_pickle("features/fm_val_graph.pkl")
fm_test_g.to_pickle("features/fm_test_graph.pkl")

print(f"fm_train_g.shape: {fm_train_g.shape}  (expect (1360850, 60))")
print(f"fm_val_g.shape:   {fm_val_g.shape}")
print(f"fm_test_g.shape:  {fm_test_g.shape}")
print()
new_cols = [c for c in fm_train_g.columns if c not in fm_train.columns]
print(f"new graph columns ({len(new_cols)}): {new_cols}")


fm_train_g.shape: (1360850, 60)  (expect (1360850, 60))
fm_val_g.shape:   (85838, 60)
fm_test_g.shape:  (88137, 59)

new graph columns (6): ['graph_degree', 'wl_K2', 'wl_K3', 'wl_K5', 'self_comment_count', 'linked_own_posts_count']


## Q13 — Retraining with graph features

In [1]:
# ----- Q13 recovery cell -----
# Run this if you restarted the kernel after Q12. Otherwise, you can skip ahead
# to q13_dt directly (fm_train_g, fm_val_g, etc. should already be in memory).

import os, gc
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ----- helpers (defensive prepare_xy matching the Part-1 version) -----
def prepare_xy(fm, drop_cols=("OwnerUserId", "timestamp", "contribution")):
    y = fm["contribution"].values.astype("int64") if "contribution" in fm.columns else None
    X = fm.drop(columns=[c for c in drop_cols if c in fm.columns]).copy()
    for c in X.columns:
        if pd.api.types.is_datetime64_any_dtype(X[c]):
            X[c] = X[c].astype("int64").astype("float64") / 1e9
            X[c] = X[c].mask(~np.isfinite(X[c]), -1)
        elif X[c].dtype == "object":
            X[c] = pd.to_numeric(X[c], errors="coerce")
        else:
            X[c] = X[c].astype("float64")
    X = X.replace([np.inf, -np.inf], np.nan)
    return X, y

def eval_split(y_true, y_pred, y_score):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "roc_auc":   roc_auc_score(y_true, y_score),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }

def fmt(metrics_tr, metrics_val):
    return {k: f"{metrics_tr[k]:.3f}/{metrics_val[k]:.3f}" for k in metrics_tr}

all_runs_g = []
def log_run_g(model_name, variant, m_tr, m_val):
    all_runs_g.append({"model": model_name, "variant": variant,
                       **{f"tr_{k}": v for k, v in m_tr.items()},
                       **{f"val_{k}": v for k, v in m_val.items()}})

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(128, 64), dropout=0.2):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

# ----- load pickled enriched feature matrices -----
fm_train_g = pd.read_pickle("features/fm_train_graph.pkl")
fm_val_g   = pd.read_pickle("features/fm_val_graph.pkl")

X_train_g, y_train_g = prepare_xy(fm_train_g)
X_val_g,   y_val_g   = prepare_xy(fm_val_g)

del fm_train_g, fm_val_g; gc.collect()

# Load Part 1 summary (for the +graph vs tabular-only comparison)
part1 = pd.read_csv("features/q7_summary.csv", index_col=0)
print("Part 1 val AUCs (from features/q7_summary.csv):")
for m in part1.index:
    print(f"  {m}: {part1.loc[m, 'roc_auc'].split('/')[1]}")

# Hyperparameter grids (same as Part 1)
dt_grid  = [{"max_depth": 5}, {"max_depth": 10}, {"max_depth": 20},
            {"max_depth": None, "min_samples_leaf": 50}]
xgb_grid = [{"max_depth": 4,  "n_estimators": 200, "learning_rate": 0.10},
            {"max_depth": 6,  "n_estimators": 200, "learning_rate": 0.10},
            {"max_depth": 8,  "n_estimators": 400, "learning_rate": 0.05}]
lr_grid  = [{"C": 0.1}, {"C": 1.0}, {"C": 10.0}]
nn_grid  = [{"hidden": (128, 64),      "dropout": 0.2, "lr": 1e-3, "epochs": 4},
            {"hidden": (256, 128, 64), "dropout": 0.3, "lr": 1e-3, "epochs": 4}]

print(f"\nX_train_g: {X_train_g.shape}, X_val_g: {X_val_g.shape}")
print(f"y_train_g pos-rate: {y_train_g.mean():.4f}")


device: cuda
Part 1 val AUCs (from features/q7_summary.csv):
  Decision Tree: 0.861
  XGBoost: 0.879
  Logistic Regression: 0.839
  Custom NN (MLP): 0.854

X_train_g: (1360850, 57), X_val_g: (85838, 57)
y_train_g pos-rate: 0.0500


In [2]:
# ----- Q13: Decision Tree -----
import gc
X_train_dt_g = X_train_g.fillna(-1).values.astype(np.float32)   # float32 halves memory
X_val_dt_g   = X_val_g.fillna(-1).values.astype(np.float32)

best_dt_g, best_dt_g_val_auc = None, -1
for params in dt_grid:
    clf = DecisionTreeClassifier(class_weight="balanced", random_state=0, **params)
    clf.fit(X_train_dt_g, y_train_g)
    s_tr  = clf.predict_proba(X_train_dt_g)[:, 1]
    s_val = clf.predict_proba(X_val_dt_g)[:, 1]
    m_tr  = eval_split(y_train_g, (s_tr  >= 0.5).astype(int), s_tr)
    m_val = eval_split(y_val_g,   (s_val >= 0.5).astype(int), s_val)
    log_run_g("DecisionTree", str(params), m_tr, m_val)
    print(f"DT {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_dt_g_val_auc:
        best_dt_g, best_dt_g_val_auc = clf, m_val["roc_auc"]
        best_dt_g_params, best_dt_g_m_tr, best_dt_g_m_val = params, m_tr, m_val

print(f"\nBEST DT_g: {best_dt_g_params}  val AUC={best_dt_g_val_auc:.3f}")
del X_train_dt_g, X_val_dt_g; gc.collect()


DT {'max_depth': 5}: val AUC=0.810, val F1=0.187
DT {'max_depth': 10}: val AUC=0.773, val F1=0.238
DT {'max_depth': 20}: val AUC=0.602, val F1=0.262
DT {'max_depth': None, 'min_samples_leaf': 50}: val AUC=0.770, val F1=0.248

BEST DT_g: {'max_depth': 5}  val AUC=0.810


0

In [3]:
# ----- Q13: XGBoost -----
import gc
pos_ratio_g = float((y_train_g == 0).sum() / (y_train_g == 1).sum())
X_train_xgb = X_train_g.values.astype(np.float32)
X_val_xgb   = X_val_g.values.astype(np.float32)

best_xgb_g, best_xgb_g_val_auc = None, -1
for params in xgb_grid:
    clf = XGBClassifier(scale_pos_weight=pos_ratio_g, tree_method="hist",
                        n_jobs=-1, random_state=0, eval_metric="logloss", **params)
    clf.fit(X_train_xgb, y_train_g)
    s_tr  = clf.predict_proba(X_train_xgb)[:, 1]
    s_val = clf.predict_proba(X_val_xgb)[:, 1]
    m_tr  = eval_split(y_train_g, (s_tr  >= 0.5).astype(int), s_tr)
    m_val = eval_split(y_val_g,   (s_val >= 0.5).astype(int), s_val)
    log_run_g("XGBoost", str(params), m_tr, m_val)
    print(f"XGB {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_xgb_g_val_auc:
        best_xgb_g, best_xgb_g_val_auc = clf, m_val["roc_auc"]
        best_xgb_g_params, best_xgb_g_m_tr, best_xgb_g_m_val = params, m_tr, m_val

print(f"\nBEST XGB_g: {best_xgb_g_params}  val AUC={best_xgb_g_val_auc:.3f}")
del X_train_xgb, X_val_xgb; gc.collect()


XGB {'max_depth': 4, 'n_estimators': 200, 'learning_rate': 0.1}: val AUC=0.865, val F1=0.294
XGB {'max_depth': 6, 'n_estimators': 200, 'learning_rate': 0.1}: val AUC=0.859, val F1=0.330
XGB {'max_depth': 8, 'n_estimators': 400, 'learning_rate': 0.05}: val AUC=0.851, val F1=0.349

BEST XGB_g: {'max_depth': 4, 'n_estimators': 200, 'learning_rate': 0.1}  val AUC=0.865


146

In [4]:
# ----- Q13: Logistic Regression -----
import gc
imp_g    = SimpleImputer(strategy="median").fit(X_train_g.values)
X_tr_imp_g = imp_g.transform(X_train_g.values).astype(np.float32)
X_va_imp_g = imp_g.transform(X_val_g.values).astype(np.float32)
scaler_g = StandardScaler().fit(X_tr_imp_g)
X_tr_sc_g = scaler_g.transform(X_tr_imp_g).astype(np.float32)
X_va_sc_g = scaler_g.transform(X_va_imp_g).astype(np.float32)
del X_tr_imp_g, X_va_imp_g; gc.collect()

best_lr_g, best_lr_g_val_auc = None, -1
for params in lr_grid:
    clf = LogisticRegression(class_weight="balanced", max_iter=2000, solver="lbfgs",
                             n_jobs=-1, random_state=0, **params)
    clf.fit(X_tr_sc_g, y_train_g)
    s_tr  = clf.predict_proba(X_tr_sc_g)[:, 1]
    s_val = clf.predict_proba(X_va_sc_g)[:, 1]
    m_tr  = eval_split(y_train_g, (s_tr  >= 0.5).astype(int), s_tr)
    m_val = eval_split(y_val_g,   (s_val >= 0.5).astype(int), s_val)
    log_run_g("LogReg", str(params), m_tr, m_val)
    print(f"LR {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_lr_g_val_auc:
        best_lr_g, best_lr_g_val_auc = clf, m_val["roc_auc"]
        best_lr_g_params, best_lr_g_m_tr, best_lr_g_m_val = params, m_tr, m_val

print(f"\nBEST LR_g: {best_lr_g_params}  val AUC={best_lr_g_val_auc:.3f}")
# keep X_tr_sc_g, X_va_sc_g around -> the NN cell reuses them


/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 3  4  5  9 11]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 3  4  5  9 11]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


LR {'C': 0.1}: val AUC=0.813, val F1=0.183
LR {'C': 1.0}: val AUC=0.809, val F1=0.201
LR {'C': 10.0}: val AUC=0.811, val F1=0.205

BEST LR_g: {'C': 0.1}  val AUC=0.813


In [5]:
# ----- Q13: Neural Net -----
import gc

def train_mlp_g(hidden, dropout, lr, epochs=4, batch_size=4096):
    torch.manual_seed(0)
    Xt = torch.tensor(X_tr_sc_g, dtype=torch.float32)
    yt = torch.tensor(y_train_g, dtype=torch.float32)
    Xv = torch.tensor(X_va_sc_g, dtype=torch.float32).to(device)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)
    model = MLP(Xt.shape[1], hidden=hidden, dropout=dropout).to(device)
    pos_w = torch.tensor([(y_train_g == 0).sum() / max((y_train_g == 1).sum(), 1)],
                          dtype=torch.float32, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            s_val = torch.sigmoid(model(Xv)).cpu().numpy()
        print(f"  epoch {ep+1}: val AUC={roc_auc_score(y_val_g, s_val):.3f}")
    return model

def score_mlp_g(model, X_np):
    model.eval()
    with torch.no_grad():
        out = []
        for i in range(0, len(X_np), 65536):
            xb = torch.tensor(X_np[i:i+65536], dtype=torch.float32, device=device)
            out.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(out)

best_nn_g, best_nn_g_val_auc = None, -1
for params in nn_grid:
    print("NN", params)
    model = train_mlp_g(**params)
    s_tr  = score_mlp_g(model, X_tr_sc_g)
    s_val = score_mlp_g(model, X_va_sc_g)
    m_tr  = eval_split(y_train_g, (s_tr  >= 0.5).astype(int), s_tr)
    m_val = eval_split(y_val_g,   (s_val >= 0.5).astype(int), s_val)
    log_run_g("MLP", str(params), m_tr, m_val)
    print(f"  -> val AUC={m_val['roc_auc']:.3f}\n")
    if m_val["roc_auc"] > best_nn_g_val_auc:
        best_nn_g, best_nn_g_val_auc = model, m_val["roc_auc"]
        best_nn_g_params, best_nn_g_m_tr, best_nn_g_m_val = params, m_tr, m_val

print(f"BEST NN_g: {best_nn_g_params}  val AUC={best_nn_g_val_auc:.3f}")
del X_tr_sc_g, X_va_sc_g; gc.collect()


NN {'hidden': (128, 64), 'dropout': 0.2, 'lr': 0.001, 'epochs': 4}
  epoch 1: val AUC=0.808
  epoch 2: val AUC=0.813
  epoch 3: val AUC=0.829
  epoch 4: val AUC=0.835
  -> val AUC=0.835

NN {'hidden': (256, 128, 64), 'dropout': 0.3, 'lr': 0.001, 'epochs': 4}
  epoch 1: val AUC=0.815
  epoch 2: val AUC=0.822
  epoch 3: val AUC=0.843
  epoch 4: val AUC=0.838
  -> val AUC=0.838

BEST NN_g: {'hidden': (256, 128, 64), 'dropout': 0.3, 'lr': 0.001, 'epochs': 4}  val AUC=0.838


0

In [6]:
# ----- Q13 results summary + +graph vs tabular-only comparison -----
summary_rows_g = [
    {"model": "Decision Tree",       **fmt(best_dt_g_m_tr,  best_dt_g_m_val)},
    {"model": "XGBoost",             **fmt(best_xgb_g_m_tr, best_xgb_g_m_val)},
    {"model": "Logistic Regression", **fmt(best_lr_g_m_tr,  best_lr_g_m_val)},
    {"model": "Custom NN (MLP)",     **fmt(best_nn_g_m_tr,  best_nn_g_m_val)},
]
summary_g = pd.DataFrame(summary_rows_g).set_index("model")
print("=== Q13 best-variant summary WITH GRAPH FEATURES (train/val) ===")
print(summary_g)

# Side-by-side with Part 1 (from disk)
part1 = pd.read_csv("features/q7_summary.csv", index_col=0)
def _val_auc(s): return float(s.split("/")[1])

p1_aucs = {m: _val_auc(part1.loc[m, "roc_auc"]) for m in part1.index}
p2_aucs = {
    "Decision Tree":       best_dt_g_m_val["roc_auc"],
    "XGBoost":             best_xgb_g_m_val["roc_auc"],
    "Logistic Regression": best_lr_g_m_val["roc_auc"],
    "Custom NN (MLP)":     best_nn_g_m_val["roc_auc"],
}
cmp = pd.DataFrame({"tabular_only": p1_aucs, "+graph": p2_aucs})
cmp["delta"] = cmp["+graph"] - cmp["tabular_only"]
print("\n=== Val ROC-AUC: tabular-only vs +graph ===")
print(cmp.round(4))

summary_g.to_csv("features/q13_summary.csv")
pd.DataFrame(all_runs_g).to_csv("features/q13_all_runs.csv", index=False)


=== Q13 best-variant summary WITH GRAPH FEATURES (train/val) ===
                        accuracy      roc_auc    precision       recall  \
model                                                                     
Decision Tree        0.836/0.856  0.925/0.810  0.218/0.111  0.879/0.590   
XGBoost              0.886/0.925  0.960/0.865  0.296/0.200  0.933/0.557   
Logistic Regression  0.871/0.822  0.922/0.813  0.252/0.105  0.807/0.710   
Custom NN (MLP)      0.882/0.925  0.949/0.838  0.285/0.171  0.894/0.432   

                              f1  
model                             
Decision Tree        0.349/0.187  
XGBoost              0.450/0.294  
Logistic Regression  0.384/0.183  
Custom NN (MLP)      0.432/0.245  

=== Val ROC-AUC: tabular-only vs +graph ===
                     tabular_only  +graph   delta
Decision Tree               0.861  0.8099 -0.0511
XGBoost                     0.879  0.8650 -0.0140
Logistic Regression         0.839  0.8129 -0.0261
Custom NN (MLP)             0

## Q14 — Top-10 with graph features

In [7]:
# ----- Q14: top-10 features after adding graph features -----
feature_names_g = list(X_train_g.columns)
def top_k_g(importances, k=10):
    idx = np.argsort(importances)[::-1][:k]
    return pd.DataFrame({"feature": [feature_names_g[i] for i in idx],
                         "importance": importances[idx]})

dt_top_g  = top_k_g(best_dt_g.feature_importances_)
xgb_top_g = top_k_g(best_xgb_g.feature_importances_)
lr_top_g  = top_k_g(np.abs(best_lr_g.coef_[0]))

print("=== DT top-10 (with graph features) ===")
print(dt_top_g.to_string(index=False))
print("\n=== XGB top-10 (with graph features) ===")
print(xgb_top_g.to_string(index=False))
print("\n=== LR top-10 (with graph features) ===")
print(lr_top_g.to_string(index=False))

# Highlight which graph features made the top 10s
graph_cols = {"graph_degree", "wl_K3", "wl_K5", "wl_K10", "wl_Kconvergence",
              "self_comment_count", "linked_own_posts_count"}
print("\n=== Graph features that landed in any top-10 ===")
in_dt  = set(dt_top_g["feature"])  & graph_cols
in_xgb = set(xgb_top_g["feature"]) & graph_cols
in_lr  = set(lr_top_g["feature"])  & graph_cols
print(f"  Decision Tree:  {sorted(in_dt)}")
print(f"  XGBoost:        {sorted(in_xgb)}")
print(f"  LogReg:         {sorted(in_lr)}")

pd.concat({"DT": dt_top_g.reset_index(drop=True),
           "XGB": xgb_top_g.reset_index(drop=True),
           "LR":  lr_top_g.reset_index(drop=True)}, axis=1
).to_csv("features/q14_top10.csv")


=== DT top-10 (with graph features) ===
                              feature  importance
                         graph_degree    0.848293
     postHistory__RevisionGUID__count    0.045278
postHistory__PostHistoryTypeId__count    0.041276
                  badges__Name__count    0.022614
      comments__ContentLicense__count    0.012084
                 badges__Class__count    0.011098
             posts__CreationDate__max    0.004817
                comments__Text__count    0.004300
          comments__CreationDate__max    0.003338
       postHistory__CreationDate__max    0.003183

=== XGB top-10 (with graph features) ===
                              feature  importance
                         graph_degree    0.251719
postHistory__PostHistoryTypeId__count    0.090701
             posts__CreationDate__min    0.071177
             posts__CreationDate__max    0.062732
                 badges__Class__count    0.054662
   postHistory__ContentLicense__count    0.043254
                  